# Matrix multiplication, CPU to CUDA: full reproduction on Colab

Runs `scripts/reproduce.sh`: build → correctness tests → benchmarks (all versions + cuBLAS) → edge sweep →
Nsight Compute profiling → summary tables → plots.

1. **Runtime → Change runtime type → T4 GPU** (or any NVIDIA GPU).
2. Edit the settings cell, then **Runtime → Run all**.
3. Results are written straight to Google Drive as they are produced, so a disconnect loses nothing.
   A zip is also downloaded at the end.

Expect roughly 20–40 minutes on a T4, mostly the single-thread CPU baseline and the profiler.

## 1. Settings

In [ ]:
# Where the code comes from:
#   "github"    clone the repo (private repo: add a Colab secret GH_TOKEN with read access to it)
#   "drive_zip" unzip MyDrive/nvidia_mm.zip (a zip of the repo folder whose top folder is nvidia_mm/)
CODE_SOURCE = "github"
REPO = "shriramiitm/nvidia-mm"

# Instructor-approved 6th benchmark shape "M K N", or "" to skip
SHAPE6 = ""

SKIP_CPU = False      # True skips the slow single-thread CPU baseline in the benchmarks
SKIP_PROFILE = False  # True skips Nsight Compute

## 2. GPU and toolchain

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,driver_version,clocks.max.sm,power.limit --format=csv
!nvcc --version | tail -n 2
!which ncu || ls /usr/local/cuda/bin | grep -i ncu || echo "ncu not found: set SKIP_PROFILE = True"

## 3. Google Drive (results are saved here as they are written)

In [ ]:
import os, time
from google.colab import drive
drive.mount('/content/drive')
RUN = time.strftime('%Y%m%d-%H%M')
RESULTS = f'/content/drive/MyDrive/nvidia_mm_results/{RUN}'
os.makedirs(RESULTS, exist_ok=True)
print('results ->', RESULTS)

## 4. Get a clean copy of the code

In [ ]:
import subprocess, shutil
os.chdir('/content')
shutil.rmtree('/content/nvidia_mm', ignore_errors=True)
if CODE_SOURCE == 'github':
    from google.colab import userdata
    try:
        tok = userdata.get('GH_TOKEN')
        url = f'https://{tok}@github.com/{REPO}.git'
    except Exception:
        url = f'https://github.com/{REPO}.git'   # public repo
    subprocess.run(['git', 'clone', '-q', url, 'nvidia_mm'], check=True)
    # don't leave the token in nvidia_mm/.git/config
    subprocess.run(['git', '-C', 'nvidia_mm', 'remote', 'set-url', 'origin', f'https://github.com/{REPO}.git'], check=True)
    print(subprocess.run(['git', '-C', 'nvidia_mm', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
else:
    subprocess.run(['unzip', '-oq', '/content/drive/MyDrive/nvidia_mm.zip', '-d', '/content'], check=True)
    shutil.rmtree('/content/__MACOSX', ignore_errors=True)
# results/ points at Drive, so every file is saved the moment it is written
shutil.rmtree('/content/nvidia_mm/results', ignore_errors=True)
os.symlink(RESULTS, '/content/nvidia_mm/results')
!ls /content/nvidia_mm

## 5. Run everything

In [ ]:
os.environ['SHAPE6'] = SHAPE6
os.environ['SKIP_CPU'] = '1' if SKIP_CPU else '0'
os.environ['SKIP_PROFILE'] = '1' if SKIP_PROFILE else '0'
!cd /content/nvidia_mm && bash scripts/reproduce.sh

## 6. Quick look

In [ ]:
!tail -n 3 /content/nvidia_mm/results/tests.txt
!grep -E 'FAIL' /content/nvidia_mm/results/tests.txt | head -n 20 || true
!column -s, -t < /content/nvidia_mm/results/results.csv | head -n 60

## 7. Download a zip of the results

In [ ]:
!cd /content/nvidia_mm && zip -qr /content/results.zip results/
from google.colab import files
files.download('/content/results.zip')